<a href="https://colab.research.google.com/github/LuisPrieto123/MINE_4210_ADL_202520/blob/main/MATI_ADL_Gr9_Proy_Mod1_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MODELO VERSION 2 PARA CLASIFICACION DE ESPECIES PICIFORMES

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os
import shutil
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy('mixed_float16')

# --- CONFIGURACIÓN DE PARÁMETROS GLOBALES ---
# Directorio que contiene las carpetas Piciformes y No_Piciformes
BASE_DATA_DIR = '/content/drive/MyDrive/MATI/ADL_PRY/MOD1'
# Nuevo directorio donde se crearán las carpetas train, validation y test
OUTPUT_DATA_DIR = '/content/drive/MyDrive/MATI/ADL_PRY/MOD1_DTOS'

# Parámetros de división de datos
TEST_SPLIT = 0.15
VALIDATION_SPLIT = 0.15
RANDOM_SEED = 42

# Parámetros del modelo y entrenamiento
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
NUM_CLASSES = 1 # Salida binaria (Sigmoid)
EPOCHS_PHASE_1 = 10
EPOCHS_PHASE_2 = 15
FINE_TUNE_AT = 100 # Congelar las primeras 100 capas del ResNet50 en Fase 2

# Parámetros del Clasificador (Head)
UNITS_DENSE_1 = 512
DROPOUT_1 = 0.5
UNITS_DENSE_2 = 256
DROPOUT_2 = 0.3

# =======================================================================
# 1. FUNCIÓN DE PREPARACIÓN Y DIVISIÓN DEL DATASET
# =======================================================================
from google.colab import drive
drive.mount('/content/drive')

def split_and_copy_data(base_dir, output_dir, test_size, val_size, seed):
    """Divide las imágenes y las copia a los directorios de salida (train/val/test)."""

    # Proporción de validación tomada del conjunto (1 - test_size)
    val_from_train_size = val_size / (1 - test_size)

    classes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

    # Crear la estructura de directorios de salida
    for split in ['train', 'validation', 'test']:
        for class_name in classes:
            os.makedirs(os.path.join(output_dir, split, class_name), exist_ok=True)

    print(f"Clases encontradas: {classes}")

    for class_name in classes:
        class_path = os.path.join(base_dir, class_name)
        all_files = [os.path.join(class_path, f) for f in os.listdir(class_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        if not all_files:
            print(f"Advertencia: No se encontraron imágenes en {class_name}")
            continue

        # 1er Split: Separar Test
        train_val_files, test_files = train_test_split(
            all_files, test_size=test_size, random_state=seed
        )

        # 2do Split: Separar Entrenamiento y Validación
        train_files, val_files = train_test_split(
            train_val_files, test_size=val_from_train_size, random_state=seed
        )

        # Copiar archivos a los nuevos directorios
        print(f"\n--- {class_name} ---")
        splits_data = {
            'train': train_files,
            'validation': val_files,
            'test': test_files
        }

        for split_name, file_list in splits_data.items():
            dest_dir = os.path.join(output_dir, split_name, class_name)
            for file_path in file_list:
                shutil.copy(file_path, dest_dir)
            print(f"Copiados {len(file_list)} archivos a {split_name}")


# -----------------------------------------------------------------------
# EJECUTAR PREPARACIÓN DE DATOS (Descomentar y ajustar la ruta si es necesario)
# -----------------------------------------------------------------------
# Asegúrate de que tu directorio BASE_DATA_DIR existe antes de ejecutar esto.
if not os.path.exists(OUTPUT_DATA_DIR):
     print("Iniciando la división y copia de archivos...")
     split_and_copy_data(BASE_DATA_DIR, OUTPUT_DATA_DIR, TEST_SPLIT, VALIDATION_SPLIT, RANDOM_SEED)
else:
     print("Directorio de datos dividido ya existe. Saltando la copia.")

import os # Importar el módulo os

# =======================================================================
# 2. CARGA DE DATOS USANDO GENERADORES
# =======================================================================

TRAIN_DIR = os.path.join(OUTPUT_DATA_DIR, 'train')
VAL_DIR = os.path.join(OUTPUT_DATA_DIR, 'validation')
TEST_DIR = os.path.join(OUTPUT_DATA_DIR, 'test')

# Generador para entrenamiento con Aumento de Datos (Data Augmentation)
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

# Generadores para validación y prueba (solo preprocesamiento)
val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)
test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)

# Creación de generadores de datos
try:
    train_generator = train_datagen.flow_from_directory(TRAIN_DIR, target_size=IMAGE_SIZE,
                                                        batch_size=BATCH_SIZE, class_mode='binary')
    validation_generator = val_datagen.flow_from_directory(VAL_DIR, target_size=IMAGE_SIZE,
                                                        batch_size=BATCH_SIZE, class_mode='binary')
    test_generator = test_datagen.flow_from_directory(TEST_DIR, target_size=IMAGE_SIZE,
                                                        batch_size=BATCH_SIZE, class_mode='binary',
                                                        shuffle=False) # Importante para la evaluación
except Exception as e:
    print(f"ERROR: No se pudieron cargar los generadores de datos. Asegúrate de que las rutas sean correctas y que la división de datos se haya ejecutado. {e}")
    exit()

print("Mapeo de Clases:", train_generator.class_indices)

# =======================================================================
# 3. DEFINICIÓN Y COMPILACIÓN DEL MODELO
# =======================================================================

def build_transfer_model():
    # Cargar ResNet50 con pesos pre-entrenados en ImageNet
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=IMAGE_SIZE + (3,))
    base_model.trainable = False # Congelar para la Fase 1

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Capa Densa 1 (Head Clasificador)
    x = Dense(UNITS_DENSE_1, activation='relu')(x)
    x = Dropout(DROPOUT_1)(x)

    # Capa Densa 2
    x = Dense(UNITS_DENSE_2, activation='relu')(x)
    x = Dropout(DROPOUT_2)(x)

    # Capa de Salida Binaria
    predictions = Dense(NUM_CLASSES, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

model = build_transfer_model()

# Compilación para la Fase 1
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy'] # Eliminado F1Score para evitar el error de forma de tensor
)

print("\n--- RESUMEN DE LA ARQUITECTURA DEL MODELO ---")
model.summary()

# =======================================================================
# 4. ENTRENAMIENTO EN DOS FASES
# =======================================================================

# --- FASE 1: ENTRENAMIENTO DEL CLASIFICADOR (HEAD) ---

print(f"\n--- INICIANDO FASE 1: ENTRENAMIENTO DEL CLASIFICADOR (HEAD) --- (Épocas: {EPOCHS_PHASE_1})")

history_phase_1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE_1,
    validation_data=validation_generator
)


# --- FASE 2: AJUSTE FINO (FINE-TUNING) ---

# 1. Descongelar las últimas capas del modelo base
model.trainable = True

for layer in model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

# 2. Re-compilar el modelo con un Learning Rate bajo
model.compile(
    optimizer=Adam(learning_rate=1e-5), # Tasa de Aprendizaje muy pequeña
    loss='binary_crossentropy',
    metrics=['accuracy'] # Eliminado F1Score para evitar el error de forma de tensor
)

print(f"\n--- INICIANDO FASE 2: AJUSTE FINO (FINE-TUNING) --- (Épocas adicionales: {EPOCHS_PHASE_2})")

history_phase_2 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE_1 + EPOCHS_PHASE_2,
    initial_epoch=history_phase_1.epoch[-1],
    validation_data=validation_generator
)

# =======================================================================
# 5. PRUEBA Y MÉTRICAS FINALES
# =======================================================================

print("\n--- EVALUACIÓN FINAL EN EL CONJUNTO DE PRUEBA ---")

# 1. Evaluación directa para obtener pérdida y precisión
# loss, accuracy, f1_score_value = model.evaluate(test_generator, verbose=1)
loss, accuracy = model.evaluate(test_generator, verbose=1)

print(f"\nResultados de la Métrica Estándar (Conjunto de Prueba):")
print(f"  Pérdida (Test): {loss:.4f}")
print(f"  Precisión (Accuracy): {accuracy:.4f}")
# print(f"  F1-Score (Test): {f1_score_value:.4f}")


# 2. Generar Predicciones y Reporte Detallado
test_generator.reset() # Resetear el generador para asegurar el orden
y_pred_prob = model.predict(test_generator, steps=test_generator.samples // test_generator.batch_size + 1)
y_pred = np.where(y_pred_prob >= 0.5, 1, 0) # Umbral de 0.5 para la clasificación binaria

# Asegurar que el tamaño de las etiquetas verdaderas coincida con las predicciones
num_test_samples = len(y_pred)
y_true = test_generator.classes[:num_test_samples]

target_names = list(test_generator.class_indices.keys())

print("\nReporte de Clasificación Detallado:")
print(classification_report(y_true, y_pred, target_names=target_names))

print("\nMatriz de Confusión (True Negatives, False Positives, False Negatives, True Positives):")
cm = confusion_matrix(y_true, y_pred)
print(cm)

# Guardar el modelo final (opcional)
model.save('clasificador_aves_piciformes.h5')